# Mundial 2026 — Predicciones en Vivo

**Secciones:** 1. Clasificaciones/Bracket · 2. Backtest (todos los partidos) · 3. Próxima fase

**Actualizar:** `download_all(force=True)` → `retrain_full()` → re-ejecutar todas las celdas.
El notebook detecta la fase actual automáticamente (grupos → octavos → ronda 16 → cuartos → semis → final).

In [1]:
import sys, json, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.models.poisson_model import PoissonModel, WC_GOAL_SCALE
from src.prediction.predict import predict_matches, format_predictions
from src.prediction.update import update_elo, save_elo, load_elo
from src.prediction.tracker import save_predictions, evaluate, summary_metrics
from src.simulation.tournament import GROUPS, compute_standings, NAME_ALIASES, determine_qualifiers

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Load models
with open('../models/poisson_params.json') as f:
    params = json.load(f)
elo_ratings = load_elo('../models/elo_ratings.json')
xgb = joblib.load('../models/xgb_pipeline.joblib')

poisson = PoissonModel()
poisson.params_ = params
poisson.teams_  = list(params['attack'].keys())

# Load raw match data
df_raw  = pd.read_csv('../data/raw/results.csv', parse_dates=['date'])
wc26    = df_raw[(df_raw['tournament'] == 'FIFA World Cup') & (df_raw['date'].dt.year == 2026)].copy()
played  = wc26.dropna(subset=['home_score', 'away_score']).copy()
pending = wc26[wc26['home_score'].isna()].copy()

# Split group stage vs knockout
ALL_GS_TEAMS = {t for teams in GROUPS.values() for t in teams}
gs_played = played[
    played['home_team'].isin(ALL_GS_TEAMS) & played['away_team'].isin(ALL_GS_TEAMS)
].copy()
ko_played = played[
    ~(played['home_team'].isin(ALL_GS_TEAMS) & played['away_team'].isin(ALL_GS_TEAMS))
].copy()

n_gs = len(gs_played)
n_ko = len(ko_played)

# Phase detection (72 GS + 16 R32 + 8 R16 + 4 QF + 2 SF + 2 Final/3rd = 104 total)
if   n_gs < 72: NEXT_PHASE = 'group_stage'
elif n_ko < 16: NEXT_PHASE = 'round_of_32'
elif n_ko < 24: NEXT_PHASE = 'round_of_16'
elif n_ko < 28: NEXT_PHASE = 'quarter_final'
elif n_ko < 30: NEXT_PHASE = 'semi_final'
else:           NEXT_PHASE = 'final'

PHASE_ES = {
    'group_stage':   'Fase de Grupos',
    'round_of_32':   'Octavos de Final',
    'round_of_16':   'Ronda de 16',
    'quarter_final': 'Cuartos de Final',
    'semi_final':    'Semifinales',
    'final':         'Final / 3er Puesto',
}

# ── Bracket helpers ─────────────────────────────────────────────────────────────

def _find_ko_winner(results_df, team_a, team_b):
    """Return the winner of a knockout match (None if not yet played)."""
    mask = (
        ((results_df['home_team'] == team_a) & (results_df['away_team'] == team_b)) |
        ((results_df['home_team'] == team_b) & (results_df['away_team'] == team_a))
    )
    rows = results_df[mask].dropna(subset=['home_score'])
    if rows.empty:
        return None
    row = rows.iloc[0]
    hs, as_ = int(row['home_score']), int(row['away_score'])
    if hs > as_: return row['home_team']
    if as_ > hs: return row['away_team']
    # Draw after 90 min → ELO tiebreak (covers ET/penalties ambiguity)
    ra = elo_ratings.get(NAME_ALIASES.get(team_a, team_a), 1500)
    rb = elo_ratings.get(NAME_ALIASES.get(team_b, team_b), 1500)
    return team_a if ra >= rb else team_b


def build_bracket(gs_df, ko_df):
    """
    Walk the seeded bracket using actual results.
    Returns dict with each round's pairs/winners and 'next_pairs' (matches to predict next).
    """
    # Build R32 pairs from final group standings
    all_standings = {
        g: compute_standings(
            gs_df[gs_df['home_team'].isin(teams) & gs_df['away_team'].isin(teams)], teams
        )
        for g, teams in GROUPS.items()
    }
    winners, runners_up, thirds = determine_qualifiers(all_standings)
    seeds = winners + runners_up + thirds           # 32 teams
    r32_pairs = [(seeds[i], seeds[31 - i]) for i in range(16)]

    bracket = {}
    current_pairs = r32_pairs
    round_keys = ['r32', 'r16', 'qf', 'sf']

    for rnd in round_keys:
        rnd_winners, rnd_losers = [], []
        complete = True
        for h, a in current_pairs:
            w = _find_ko_winner(ko_df, h, a)
            if w is None:
                complete = False
                break
            rnd_winners.append(w)
            rnd_losers.append(a if w == h else h)

        bracket[rnd] = {
            'pairs':   current_pairs,
            'winners': rnd_winners if complete else [],
            'losers':  rnd_losers  if complete else [],
        }
        if not complete:
            bracket['next_pairs'] = [
                (h, a) for h, a in current_pairs if _find_ko_winner(ko_df, h, a) is None
            ]
            return bracket

        current_pairs = [
            (rnd_winners[i * 2], rnd_winners[i * 2 + 1])
            for i in range(len(rnd_winners) // 2)
        ]

    # SF complete → Final + 3rd-place match
    sf_losers = bracket.get('sf', {}).get('losers', [])
    bracket['final'] = {'pairs': current_pairs}
    bracket['3rd']   = {'pairs': [tuple(sf_losers)] if len(sf_losers) == 2 else []}
    bracket['next_pairs'] = current_pairs + bracket['3rd']['pairs']
    return bracket


print(f'Partidos jugados   : {len(played)}  (GS: {n_gs}, Eliminatoria: {n_ko})')
print(f'Siguiente fase     : {PHASE_ES[NEXT_PHASE]}')

Partidos jugados   : 73  (GS: 73, Eliminatoria: 0)
Siguiente fase     : Octavos de Final


## 1. Estado actual — Clasificaciones

In [2]:
if n_gs < 72:
    # Group stage in progress — show standings per group
    max_pj = max(
        (int(compute_standings(
            gs_played[gs_played['home_team'].isin(t) & gs_played['away_team'].isin(t)], t
        )[['w','d','l']].sum(axis=1).max()) for t in GROUPS.values()),
        default=0
    )
    print(f'Clasificaciones — tras Jornada {max_pj} (partidos con resultado)\n')
    for group, teams in sorted(GROUPS.items()):
        gm = gs_played[gs_played['home_team'].isin(teams) & gs_played['away_team'].isin(teams)]
        st = compute_standings(gm, teams)
        print(f'Grupo {group}')
        for rank, (_, r) in enumerate(st.iterrows(), 1):
            pj = int(r['w'] + r['d'] + r['l'])
            if rank <= 2 and pj == 3:
                marker = ' ✓ CLASIFICA'
            elif rank <= 2:
                marker = ' (proyectado)'
            else:
                marker = ''
            print(f'  {rank}. {r["team"]:26s}  {int(r["pts"]):2d}pts  {int(r["gf"]):2d}:{int(r["ga"]):2d}  ({pj}PJ){marker}')
        print()
else:
    # Knockout phase — show qualified teams (final group standings, collapsed)
    print('=== Fase de grupos finalizada ===\n')
    all_standings = {
        g: compute_standings(
            gs_played[gs_played['home_team'].isin(t) & gs_played['away_team'].isin(t)], t
        )
        for g, t in GROUPS.items()
    }
    wnrs, rnrs, thirds = determine_qualifiers(all_standings)
    print('Ganadores de grupo (semillas 1–12):')
    for i, (g, w) in enumerate(zip(sorted(GROUPS), wnrs), 1):
        print(f'  {i:2d}. [{g}] {w}')
    print('\nSegundos (semillas 13–24):')
    for i, (g, r) in enumerate(zip(sorted(GROUPS), rnrs), 13):
        print(f'  {i:2d}. [{g}] {r}')
    print('\nMejores terceros (semillas 25–32):')
    for i, t in enumerate(thirds, 25):
        print(f'  {i:2d}. {t}')
    if n_ko > 0:
        print(f'\nPartidos de eliminatoria jugados: {n_ko}')
        print(f'Fase actual: {PHASE_ES[NEXT_PHASE]}')

=== Fase de grupos finalizada ===

Ganadores de grupo (semillas 1–12):
   1. [A] Argentina
   2. [B] United States
   3. [C] Belgium
   4. [D] Switzerland
   5. [E] Brazil
   6. [F] Spain
   7. [G] Colombia
   8. [H] England
   9. [I] Germany
  10. [J] Mexico
  11. [K] France
  12. [L] Netherlands

Segundos (semillas 13–24):
  13. [A] Austria
  14. [B] Australia
  15. [C] Egypt
  16. [D] Canada
  17. [E] Morocco
  18. [F] Cape Verde
  19. [G] Portugal
  20. [H] Croatia
  21. [I] Ivory Coast
  22. [J] South Africa
  23. [K] Norway
  24. [L] Japan

Mejores terceros (semillas 25–32):
  25. DR Congo
  26. Sweden
  27. Ghana
  28. Ecuador
  29. Bosnia and Herzegovina
  30. Algeria
  31. Paraguay
  32. Senegal


## 2. Backtest completo — todos los partidos jugados

In [3]:
def _backtest(df):
    if len(df) == 0:
        return None
    p = predict_matches(list(zip(df['home_team'], df['away_team'])), poisson, xgb, elo_ratings, neutral=True)
    return summary_metrics(evaluate(p, df))

# Group stage matchday segments
md1 = gs_played[gs_played['date'].dt.date <= pd.Timestamp('2026-06-17').date()]
md2 = gs_played[(gs_played['date'].dt.date >= pd.Timestamp('2026-06-18').date()) &
                (gs_played['date'].dt.date <= pd.Timestamp('2026-06-22').date())]
md3 = gs_played[gs_played['date'].dt.date >= pd.Timestamp('2026-06-23').date()]

# Knockout round segments (by match count: 16, 8, 4, 2, 2)
ko_labels = ['Octavos (R32)', 'Ronda de 16', 'Cuartos', 'Semis', 'Final/3er']
ko_sizes  = [16, 8, 4, 2, 2]
ko_segs   = []
start = 0
for lbl, sz in zip(ko_labels, ko_sizes):
    chunk = ko_played.iloc[start:start + sz]
    if len(chunk) > 0:
        ko_segs.append((lbl, chunk))
    start += sz

segments = []
if len(md1) > 0: segments.append(('Jornada 1  (11-17 jun)', md1))
if len(md2) > 0: segments.append(('Jornada 2  (18-22 jun)', md2))
if len(md3) > 0: segments.append(('Jornada 3  (23-27 jun)', md3))
segments.extend(ko_segs)
segments.append(('Total', played))

print(f"{'':22s} {'N':>4}  {'Resultado':>10}  {'Score exacto':>13}  {'MAE goles':>9}")
print('-' * 66)
for label, df_seg in segments:
    m = _backtest(df_seg)
    if not m or m.get('n_matches', 0) == 0:
        continue
    n   = m['n_matches']
    ra  = m['result_accuracy']
    ep  = m['exact_score_pct']
    mae = (m['home_goal_mae'] + m['away_goal_mae']) / 2
    print(f"{label:22s} {n:>4d}  {ra:>8.1%} ({int(ra*n):2d}/{n})  {ep:>8.1%} ({int(ep*n):2d}/{n})  {mae:>8.2f}")

                          N   Resultado   Score exacto  MAE goles
------------------------------------------------------------------


Jornada 1  (11-17 jun)   24     62.5% (15/24)     29.2% ( 7/24)      0.73


Jornada 2  (18-22 jun)   20     80.0% (16/20)     15.0% ( 3/20)      0.68


Jornada 3  (23-27 jun)   29     75.9% (22/29)     13.8% ( 4/29)      0.76


Total                    73     72.6% (53/73)     19.2% (14/73)      0.73


In [4]:
all_pairs = list(zip(played['home_team'], played['away_team']))
preds_all = predict_matches(all_pairs, poisson, xgb, elo_ratings, neutral=True)
ev_all    = evaluate(preds_all, played)

dates = played[['home_team', 'away_team', 'date']].copy()
ev_all = ev_all.merge(dates, on=['home_team', 'away_team'], how='left')

def _jornada(d):
    dt = d.date()
    if dt <= pd.Timestamp('2026-06-17').date(): return 'J1'
    if dt <= pd.Timestamp('2026-06-22').date(): return 'J2'
    if dt <= pd.Timestamp('2026-06-27').date(): return 'J3'
    return 'KO'

_out_label = {'home_win': 'H', 'draw': 'D', 'away_win': 'A'}

ev_all['J']      = ev_all['date'].apply(_jornada)
ev_all['Real']   = ev_all['home_score'].astype(int).astype(str) + '-' + ev_all['away_score'].astype(int).astype(str)
ev_all['Out']    = ev_all['pred_result'].map(_out_label)
ev_all['Res']    = ev_all['result_correct'].map({True: '✓', False: '✗'})
ev_all['Sc']     = ev_all['exact_score_correct'].map({True: '✓', False: '✗'})
ev_all['P(loc)'] = ev_all['p_home_win'].map('{:.0%}'.format)
ev_all['P(emp)'] = ev_all['p_draw'].map('{:.0%}'.format)
ev_all['P(vis)'] = ev_all['p_away_win'].map('{:.0%}'.format)

display_cols = ['J', 'home_team', 'away_team', 'Real', 'pred_score', 'Out', 'Res', 'Sc',
                'P(loc)', 'P(emp)', 'P(vis)', 'result_correct']
out_df = ev_all[display_cols].rename(columns={
    'home_team': 'Local', 'away_team': 'Visitante', 'pred_score': 'Pred'
})

def _row_color(row):
    bg = '#d4edda' if row['result_correct'] else '#f8d7da'
    return [f'background-color: {bg}; color: #1a1a1a'] * len(row)

display(
    out_df.style
    .apply(_row_color, axis=1)
    .hide(['result_correct'], axis='columns')
    .set_properties(**{
        'font-size': '13px',
        'text-align': 'center',
        'color': '#1a1a1a',
        'padding': '4px 8px',
    })
    .set_properties(subset=['Local', 'Visitante'], **{'text-align': 'left', 'font-weight': '500'})
    .set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#1f1f1f'),
            ('color', 'white'),
            ('font-size', '13px'),
            ('padding', '6px 8px'),
        ]},
        {'selector': 'table', 'props': [('border-collapse', 'collapse')]},
    ])
    .hide(axis='index')
)
print('\nJ = jornada (J1/J2/J3) o KO (eliminatoria) | Pred = score Poisson | Out = resultado ensemble | Res evalúa Out')

J,Local,Visitante,Real,Pred,Out,Res,Sc,P(loc),P(emp),P(vis)
J1,Mexico,South Africa,2-0,2-0,H,✓,✓,77%,17%,6%
J1,South Korea,Czech Republic,2-1,2-1,H,✓,✓,56%,29%,14%
J1,Canada,Bosnia and Herzegovina,1-1,3-0,H,✗,✗,76%,21%,3%
J1,United States,Paraguay,4-1,2-2,H,✓,✗,42%,23%,35%
J1,Qatar,Switzerland,1-1,1-5,A,✗,✗,3%,32%,65%
J1,Brazil,Morocco,1-1,2-1,H,✗,✗,43%,41%,16%
J1,Haiti,Scotland,0-1,1-2,A,✓,✗,19%,23%,57%
J1,Australia,Turkey,2-0,1-1,H,✓,✗,55%,23%,22%
J1,Germany,Curaçao,7-1,6-1,H,✓,✗,94%,4%,2%
J1,Ivory Coast,Ecuador,1-0,0-1,D,✗,✗,19%,49%,32%



J = jornada (J1/J2/J3) o KO (eliminatoria) | Pred = score Poisson | Out = resultado ensemble | Res evalúa Out


## 3. Próxima fase — predicciones

In [5]:
MAX_G = 5

# ── Determine next round fixture pairs ─────────────────────────────────────────
is_knockout = (NEXT_PHASE != 'group_stage')

if not is_knockout:
    all_pending = pending.sort_values('date').copy()
    next_pairs  = list(zip(all_pending['home_team'], all_pending['away_team']))
    save_phase  = 'group_stage'
    save_md     = n_gs // 24   # 0 pre-J1, 1 pre-J2, 2 pre-J3
else:
    bracket    = build_bracket(gs_played, ko_played)
    next_pairs = bracket.get('next_pairs', [])
    save_phase = NEXT_PHASE
    save_md    = 0

phase_title = PHASE_ES[NEXT_PHASE]
print(f'=== {phase_title} — {len(next_pairs)} partido(s) ===\n')

if not next_pairs:
    print('No hay partidos pendientes. Torneo finalizado.')
else:
    preds_next = predict_matches(next_pairs, poisson, xgb, elo_ratings, neutral=True)

    rows = []
    for idx_m, ((home, away), (_, pred_row)) in enumerate(zip(next_pairs, preds_next.iterrows())):
        h_model = NAME_ALIASES.get(home, home)
        a_model = NAME_ALIASES.get(away, away)
        mat  = poisson.predict_score_matrix(h_model, a_model, neutral=True, max_goals=MAX_G, goal_scale=WC_GOAL_SCALE)
        flat = mat.flatten()
        top_idxs = np.argsort(flat)[::-1][:5]

        # Date label: available for group stage pending, match number for knockout
        if not is_knockout:
            date_rows = all_pending[(all_pending['home_team'] == home) & (all_pending['away_team'] == away)]
            fecha = pd.Timestamp(date_rows['date'].iloc[0]).strftime('%d %b') if len(date_rows) > 0 else '—'
        else:
            fecha = f'#{idx_m + 1}'

        entry = {'Fecha': fecha, 'Local': home, 'Visitante': away}
        for rank, idx in enumerate(top_idxs, 1):
            h_g, a_g = idx // (MAX_G + 1), idx % (MAX_G + 1)
            entry[f'#{rank}'] = f'{h_g}-{a_g} ({flat[idx]:.0%})'

        p_loc = pred_row['p_home_win']
        p_emp = pred_row['p_draw']
        p_vis = pred_row['p_away_win']
        entry.update({
            'p_home': p_loc, 'p_draw': p_emp, 'p_away': p_vis,
            'P(loc)': f'{p_loc:.0%}',
            'P(emp)': f'{p_emp:.0%}',
            'P(vis)': f'{p_vis:.0%}',
        })
        rows.append(entry)

    pred_df = pd.DataFrame(rows)

    def _pred_row_color(row):
        max_p = max(row['p_home'], row['p_draw'], row['p_away'])
        if max_p >= 0.60:   bg = '#d4edda'
        elif max_p >= 0.45: bg = '#fff3cd'
        else:               bg = '#ffffff'
        return [f'background-color: {bg}; color: #1a1a1a'] * len(row)

    display_cols = ['Fecha', 'Local', 'Visitante', '#1', '#2', '#3', '#4', '#5',
                    'P(loc)', 'P(emp)', 'P(vis)', 'p_home', 'p_draw', 'p_away']
    show_df = pred_df[display_cols]

    display(
        show_df.style
        .apply(_pred_row_color, axis=1)
        .hide(['p_home', 'p_draw', 'p_away'], axis='columns')
        .set_properties(**{
            'font-size': '13px',
            'text-align': 'center',
            'color': '#1a1a1a',
            'padding': '4px 8px',
        })
        .set_properties(subset=['Local', 'Visitante'], **{'text-align': 'left', 'font-weight': '500'})
        .set_table_styles([
            {'selector': 'th', 'props': [
                ('background-color', '#1f1f1f'),
                ('color', 'white'),
                ('font-size', '13px'),
                ('padding', '6px 8px'),
            ]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse')]},
        ])
        .hide(axis='index')
    )

    if is_knockout:
        print('Eliminatoria: P(emp) = prob. de empate al 90\'. P(loc)/P(vis) incluyen ET y penaltis.')
    else:
        print('Verde = favorito claro (>60%) · Amarillo = leve ventaja (45-60%) · Blanco = parejo')

    save_predictions(preds_next, phase=save_phase, matchday=save_md)

=== Octavos de Final — 16 partido(s) ===



Fecha,Local,Visitante,#1,#2,#3,#4,#5,P(loc),P(emp),P(vis)
#1,Argentina,Senegal,2-0 (15%),3-0 (14%),4-0 (11%),1-0 (9%),2-1 (8%),76%,12%,12%
#2,United States,Paraguay,1-1 (12%),2-1 (9%),1-2 (8%),2-2 (7%),1-0 (6%),42%,23%,35%
#3,Belgium,Algeria,1-1 (10%),2-1 (10%),2-0 (8%),3-1 (7%),2-2 (6%),52%,29%,18%
#4,Switzerland,Bosnia and Herzegovina,4-0 (16%),3-0 (16%),5-0 (13%),2-0 (11%),4-1 (7%),88%,10%,1%
#5,Brazil,Ecuador,0-0 (22%),1-0 (18%),1-1 (14%),0-1 (11%),2-0 (9%),51%,34%,15%
#6,Spain,Ghana,2-0 (19%),3-0 (16%),1-0 (14%),4-0 (10%),2-1 (7%),71%,26%,3%
#7,Colombia,Sweden,3-1 (9%),3-0 (9%),4-1 (8%),4-0 (8%),2-1 (8%),72%,19%,9%
#8,England,DR Congo,1-0 (21%),0-0 (17%),2-0 (15%),1-1 (11%),2-1 (7%),53%,35%,12%
#9,Germany,Japan,1-1 (11%),1-2 (9%),2-1 (8%),2-2 (7%),1-3 (5%),29%,25%,46%
#10,Mexico,Norway,2-0 (10%),1-1 (10%),2-1 (10%),1-0 (8%),3-0 (8%),51%,21%,27%


Eliminatoria: P(emp) = prob. de empate al 90'. P(loc)/P(vis) incluyen ET y penaltis.
  Predictions saved -> data\predictions\round_of_32_md0.csv


## Cómo usar — una sola receta para todo el torneo

### Después de cada matchday o fase (grupos → octavos → ronda 16 → … → final)
```python
# 1. Descargar datos nuevos
from src.data.download import download_all
download_all(force=True)

# 2a. Entre jornadas de grupos (solo actualizar ELO — más rápido)
import pandas as pd
from src.prediction.update import update_elo, save_elo, load_elo
df = pd.read_csv('data/raw/results.csv', parse_dates=['date'])
nuevos = df[(df['tournament'] == 'FIFA World Cup') & (df['date'].dt.year == 2026)].dropna(subset=['home_score'])
elo = update_elo(nuevos, load_elo()); save_elo(elo)

# 2b. Entre fases (reentrenamiento completo, ~2 min — recomendado)
from src.prediction.update import retrain_full
retrain_full()

# 3. Re-ejecutar el notebook — la sección 3 se adapta automáticamente
```

### Detección automática de fase

| Partidos WC 2026 con resultado | Sección 3 muestra |
|---|---|
| 0 – 71 | Próximos partidos de fase de grupos |
| 72 | **Octavos de Final** (16 partidos derivados de clasificaciones) |
| 73 – 87 | Octavos pendientes |
| 88 | **Ronda de 16** (8 partidos derivados de ganadores R32) |
| 96 | **Cuartos de Final** (4 partidos) |
| 100 | **Semifinales** (2 partidos) |
| 102 | **Final + 3er puesto** (2 partidos) |

### Predecir un partido específico
```python
from src.prediction.predict import predict_matches
import json, joblib
from src.models.poisson_model import PoissonModel
from src.prediction.update import load_elo

with open('models/poisson_params.json') as f: params = json.load(f)
poisson = PoissonModel(); poisson.params_ = params; poisson.teams_ = list(params['attack'].keys())
xgb = joblib.load('models/xgb_pipeline.joblib')
elo = load_elo()

pred = predict_matches([('France', 'Argentina')], poisson, xgb, elo, neutral=True)
print(pred.to_string(index=False))
```